In [ ]:
FIELDS = ['dr_no','date_occ','time_occ','area_name','crm_cd_desc','vict_age','vict_sex','vict_descent','premis_desc','location','cross_street','lat','lon','mocodes']
print('Config OK')

In [ ]:
import requests
base_url = 'https://data.lacity.org/resource/d5tf-ez2w.json'
select_fields = ','.join(FIELDS)
all_records = []
offset = 0
while True:
    url = base_url + '?$limit=50000&$offset=' + str(offset) + '&$select=' + select_fields + '&$where=date_occ%3E%272022-01-01T00%3A00%3A00%27&$order=date_occ%20DESC'
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    batch = r.json()
    if not batch:
        break
    all_records.extend(batch)
    print(len(all_records), 'records...')
    if len(batch) < 50000:
        break
    offset += 50000
print('Total:', len(all_records))

In [ ]:
rows = [tuple(str(rec.get(f, '') or '') for f in FIELDS) for rec in all_records]
print('Rows:', len(rows))

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
schema = StructType([StructField(f, StringType(), True) for f in FIELDS])
df = spark.createDataFrame(rows, schema)
print('DF:', df.count())

In [ ]:
df2 = df.withColumn('crash_year', F.year(F.to_date('date_occ')))
df2.write.format('delta').mode('overwrite').partitionBy('crash_year').saveAsTable('bronze_crashes')
spark.sql('SELECT crash_year, count(*) cnt FROM bronze_crashes GROUP BY 1 ORDER BY 1').show()
print('DONE:', spark.table('bronze_crashes').count())

In [ ]:
print('BRONZE COMPLETE')